In [1]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
oak_zoning_gdf = gpd.read_file("raw_data/Zoning_20261007.geojson") #Oakland zoning data
mtc_sb79_zones_gdf = gpd.read_file("raw_data/mtc_sb79_tod_zones.geojson") #MTC SB79 tier 1 and 2 zone data
oak_parcels_gdf = gpd.read_file("raw_data/oak_parcels.geojson") #Oakland parcel data

ERROR 1: PROJ: proj_create_from_database: Open of /srv/conda/envs/notebook/share/proj failed


In [3]:
#dissolve duplicate APN polygons into a single multi-part geometry per APN so each row is one parcel
oak_parcels_gdf = oak_parcels_gdf.dissolve(by="APN", as_index=False)

In [4]:
if oak_zoning_gdf.crs != mtc_sb79_zones_gdf.crs: #Check if zoning data and mtc data is in the same crs
    oak_zoning_gdf = oak_zoning_gdf.to_crs(mtc_sb79_zones_gdf.crs) #If not the same, change the zoning crs to the mtc crs

if oak_parcels_gdf.crs != mtc_sb79_zones_gdf.crs: #Check if parcel data and mtc data is in the same crs
    oak_parcels_gdf = oak_parcels_gdf.to_crs(mtc_sb79_zones_gdf.crs) #If not the same, change the parcel crs to the mtc crs

In [5]:
keep_parcel_columns = [
    'APN', #Assessor's Parcel Number
    'APN_SORT', #Assessor's Parcel Number without dashes
    'CENTROID_X', 
    'CENTROID_Y', 
    'Land', #Assessed land value (excluding structure value)
    'Imps', #Value of improvements (structure/infrastructure) on the land
    'UseCode', #Current land use (not what's zoned)
    'SitusAddress', #Address
    'geometry'
]
oak_parcels_gdf = oak_parcels_gdf[keep_parcel_columns] #Keep only the columns necessary to this project to improve processeing speeds

In [6]:
#chnage centroid column names. They are the x and y coordinates of each parcels centroid
oak_parcels_gdf = oak_parcels_gdf.rename(
    columns={"CENTROID_X": "parcel_centroid_x", "CENTROID_Y": "parcel_centroid_y"}
)

In [7]:
def get_dominant_clip(parcels_gdf, attribute_gdf):
    
    #create a temportary unique ID in case of duplicate parcels
    gdf = parcels_gdf.copy() #ensure not modifying parcels_gdf in-place
    gdf['temp_id'] = gdf.index

    #overaly the parcels with the attribute gdf their intersection
    clipped = gpd.overlay(gdf, attribute_gdf, how="intersection")

    #get the area of each parcel - reproject to CA State Place Zone 3 (EPSG: 2227)
    #there are now duplicate parcels because they were assigned to multiple zones
    clipped["overlap_area"] = clipped.to_crs(epsg=2227).area

    #take only the parcel with the larger area in the SB79 zone
    dominant_clip = (
        clipped.sort_values("overlap_area", ascending=False)
        .drop_duplicates(subset=["temp_id"])
    )

    attr_cols = [
        col for col in dominant_clip.columns 
        if col not in parcels_gdf.columns or col == "temp_id"
    ]
    
    return dominant_clip[attr_cols]

In [8]:
oak_parcels = oak_parcels_gdf.copy()

#set the temp_id to compare the clipped versions to
oak_parcels["temp_id"] = oak_parcels.index 

In [9]:
#run function to clip parcels to the MTC SB79 zones
oak_parcel_tiers = get_dominant_clip(oak_parcels, mtc_sb79_zones_gdf)

In [10]:
#run function to clip parcels to zoning data
oak_parcel_zones = get_dominant_clip(oak_parcels, oak_zoning_gdf)

In [11]:
#merge original parcels with parcel tiers
oak_parcels = oak_parcels.merge(
    oak_parcel_tiers,
    on="temp_id",
    how="inner" #drop parcels that don't have a tier match
)

#prevent duplicate column names
oak_parcels = oak_parcels.rename(columns={
    "zone_id": "tier_zone_id", 
    "zone_label": "tier_zone_label"
})

#keep only 3 relevant columns from the zonign data and temp_id to merge
oak_parcels = oak_parcels.merge(
    oak_parcel_zones[["temp_id", "basezone", "overlay", "znlabel"]],
    on="temp_id",
    how="left" #keep all the parcels
)

#drop temp_id because clipping and merging is done
oak_parcels = oak_parcels.drop(columns=["temp_id"])

In [12]:
oak_parcels.head(2)

,APN,APN_SORT,parcel_centroid_x,parcel_centroid_y,Land,Imps,UseCode,SitusAddress,geometry,OBJECTID,tier_zone_id,tier_zone_label,overlap_area,basezone,overlay,znlabel
0,1-1111-1,001 111100100,-1.361081e+07,4.550375e+06,161934,438407,3900,351 OAK ST OAKLAND 94607,"POLYGON ((-122.2683 37.7945, -122.26737 37.794...",149,d5a66164-5576-4a8b-9f19-a7a433b5712c,Tier 1 - Quarter Mile,32504.953681,C-45,,C-45
1,1-1111-10,001 111101000,-1.361081e+07,4.550375e+06,156615,365435,7301,311 OAK ST 105 OAKLAND 94607,"POLYGON ((-122.2683 37.7945, -122.26737 37.794...",149,d5a66164-5576-4a8b-9f19-a7a433b5712c,Tier 1 - Quarter Mile,32504.953681,C-45,,C-45


In [112]:
#save cleaned and joined gdf as a parqet file to use for analysis
oak_parcels.to_parquet("clean_data/oak_parcels_sb79tiers_zones.parquet")